In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

LAST_NAME = "Pena"
STUDENT_ID = "TUPM-26-1140"

id_digits = "".join(d for d in STUDENT_ID if d.isdigit())
assert id_digits, "Student ID must contain digits."

SEED_DIGIT = int(id_digits[-1])
ID_SUM = sum(map(int, id_digits))
NAME_LENGTH = len(LAST_NAME)
RANDOM_SEED = int(id_digits) % (2**32)

verification_value = ID_SUM * NAME_LENGTH + SEED_DIGIT

np.set_printoptions(precision=4, suppress=True)

print("Student:", LAST_NAME, STUDENT_ID)
print("Seed Digit:", SEED_DIGIT)
print("ID Sum:", ID_SUM)
print("Name Length:", NAME_LENGTH)
print("Random Seed:", RANDOM_SEED)
print("Verification Value:", verification_value)

Student: Pena TUPM-26-1140
Seed Digit: 0
ID Sum: 14
Name Length: 4
Random Seed: 261140
Verification Value: 56


In [ ]:
from pathlib import Path

WORK_DIR = Path.cwd()
DATA_DIR = WORK_DIR / "data"
OUTPUT_DIR = WORK_DIR / "output"

DATA_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)

measurement_path = DATA_DIR / "secom.data"
label_path = DATA_DIR / "secom_labels.data"
description_path = DATA_DIR / "secom.names"

for path in [measurement_path, label_path, description_path]:
    assert path.is_file(), f"Missing file: {path}"

print("Workspace:", WORK_DIR.resolve())
print("All dataset files verified.")

Workspace: C:\Users\maryg\Lab7_NumPy_Pandas_Pena\Lab7_NumPy_Pandas_Pena
All dataset files verified.


In [ ]:
demo = np.array([2, 4, 6, 8])

print("Sample Array:", demo)
print("Element-wise Multiplication:", demo * 3)

raw_measurements = np.genfromtxt(
    measurement_path, dtype=float
)

assert raw_measurements.ndim == 2

print("Shape:", raw_measurements.shape)
print("Dimensions:", raw_measurements.ndim)
print("Total Elements:", raw_measurements.size)
print("Data Type:", raw_measurements.dtype)
print("Missing Values:", np.isnan(raw_measurements).sum())

Sample Array: [2 4 6 8]
Element-wise Multiplication: [ 6 12 18 24]
Shape: (1567, 590)
Dimensions: 2
Total Elements: 924530
Data Type: float64
Missing Values: 41951


In [ ]:
block = raw_measurements[6:9, :8]
snapshot = block.copy()
block_copy = block.copy()

print("First Record:", block[0, :])
print("First Feature:", block[:, 0])
print("Last Two Records:\n", block[-2:, :])

reshaped = block_copy.reshape(3, 8)

print("Selected Block Shape:", block.shape)
print("Copied Block Shape:", block_copy.shape)
print("Reshaped Shape:", reshaped.shape)
print("Transposed Shape:", block.T.shape)
print("Flattened Size:", block.flatten().size)

block_copy[0, 0] = 999.0

print("Original Unchanged:",
      np.array_equal(block, snapshot, equal_nan=True))

First Record: [3030.27   2430.12   2230.4222 1463.6606    0.8294  100.      102.3433
    0.1247]
First Feature: [3030.27 3058.88 2967.68]
Last Two Records:
 [[3058.88   2690.15   2248.9    1004.4692    0.7884  100.      106.24
     0.1185]
 [2967.68   2600.47   2248.9    1004.4692    0.7884  100.      106.24
     0.1185]]
Selected Block Shape: (3, 8)
Copied Block Shape: (3, 8)
Reshaped Shape: (3, 8)
Transposed Shape: (8, 3)
Flattened Size: 24
Original Unchanged: True


In [ ]:
missing = np.isnan(raw_measurements)
observed = (~missing).sum(axis=0)

all_missing = observed == 0

observed_min = np.where(
    missing, np.inf, raw_measurements
).min(axis=0)

observed_max = np.where(
    missing, -np.inf, raw_measurements
).max(axis=0)

constant = (
    (observed > 0) &
    (observed_min == observed_max)
)

usable_indices = np.flatnonzero(
    ~(all_missing | constant)
)

raw_usable = raw_measurements[:, usable_indices].copy()

feature_means = np.nanmean(raw_usable, axis=0)

feature_sds = np.nanstd(
    raw_usable, axis=0, ddof=1
)

print("Total Missing Values:", missing.sum())
print("Entirely Missing Features:", all_missing.sum())
print("Constant Features:", constant.sum())
print("Retained Features:", len(usable_indices))
print("First Feature Mean:", f"{feature_means[0]:.4f}")
print("First Feature Sample SD:", f"{feature_sds[0]:.4f}")

Total Missing Values: 41951
Entirely Missing Features: 0
Constant Features: 116
Retained Features: 474
First Feature Mean: 3014.4529
First Feature Sample SD: 73.6218


In [ ]:
missing_before = np.isnan(raw_usable).sum()

feature_medians = np.nanmedian(
    raw_usable, axis=0
)

prepared_np = np.where(
    np.isnan(raw_usable),
    feature_medians,
    raw_usable
)

means = prepared_np.mean(axis=0)
sds = prepared_np.std(axis=0, ddof=1)

safe_sds = np.where(sds > 0, sds, 1.0)

standardized_np = (
    prepared_np - means
) / safe_sds

flags = np.abs(standardized_np) > 3

print("Missing Before:", missing_before)
print("Missing After:", np.isnan(prepared_np).sum())
print("First Feature Median:", f"{feature_medians[0]:.4f}")
print("Standardized Mean:",
      f"{standardized_np[:, 0].mean():.4f}")
print("Standardized SD:",
      f"{standardized_np[:, 0].std(ddof=1):.4f}")
print("Values Beyond ±3:", np.count_nonzero(flags))

Missing Before: 41136
Missing After: 0
First Feature Median: 3011.4900
Standardized Mean: 0.0000
Standardized SD: 1.0000
Values Beyond ±3: 6807


In [ ]:
n_features = raw_measurements.shape[1]

feature_names = [
    f"Feature_{i:03d}"
    for i in range(n_features)
]

df = pd.DataFrame(
    raw_measurements.copy(),
    columns=feature_names
)

labels = np.loadtxt(
    label_path,
    usecols=0,
    dtype=int
)

assert len(labels) == len(df)
assert np.isin(labels, [-1, 1]).all()

df["Result"] = labels

df["Outcome"] = np.where(
    labels == -1,
    "Pass",
    "Fail"
)

print("DataFrame Shape:", df.shape)
print("Measurement Features:", n_features)
print("Pass Records:", (labels == -1).sum())
print("Fail Records:", (labels == 1).sum())

DataFrame Shape: (1567, 592)
Measurement Features: 590
Pass Records: 1463
Fail Records: 104


In [ ]:
df.info(verbose=False)

print(
    df[feature_names[:3]]
    .describe()
    .round(4)
)

print(
    "Missing Values:",
    df[feature_names].isna().sum().sum()
)

print(
    "Outcome Counts:\n",
    df["Outcome"].value_counts()
)

print("Using iloc:\n", df.iloc[:3, :4])

print(
    "Using loc:\n",
    df.loc[:2, [feature_names[0], "Outcome"]]
)

print("Series:", type(df[feature_names[0]]))

print(
    "DataFrame:",
    type(df[[feature_names[0]]])
)

print("Selected Feature:", feature_names[0])

<class 'pandas.DataFrame'>
RangeIndex: 1567 entries, 0 to 1566
Columns: 592 entries, Feature_000 to Outcome
dtypes: float64(590), int64(1), str(1)
memory usage: 7.1 MB
       Feature_000  Feature_001  Feature_002
count    1561.0000    1560.0000    1553.0000
mean     3014.4529    2495.8502    2200.5473
std        73.6218      80.4077      29.5132
min      2743.2400    2158.7500    2060.6600
25%      2966.2600    2452.2475    2181.0444
50%      3011.4900    2499.4050    2201.0667
75%      3056.6500    2538.8225    2218.0555
max      3356.3500    2846.4400    2315.2667
Missing Values: 41951
Outcome Counts:
 Outcome
Pass    1463
Fail     104
Name: count, dtype: int64
Using iloc:
    Feature_000  Feature_001  Feature_002  Feature_003
0      3030.93      2564.00    2187.7333    1411.1265
1      3095.78      2465.14    2230.4222    1463.6606
2      2932.61      2559.94    2186.4111    1698.0172
Using loc:
    Feature_000 Outcome
0      3030.93    Pass
1      3095.78    Pass
2      2932.61    

In [ ]:
without_all_missing = df.dropna(
    axis=1,
    how="all"
)

retained_features = [
    feature_names[i]
    for i in usable_indices
]

clean = without_all_missing[
    retained_features
].copy()

clean = clean.fillna(clean.median())

clean["Result"] = df["Result"]
clean["Outcome"] = df["Outcome"]

clean["Is_Fail"] = (
    clean["Result"] == 1
)

match = np.allclose(
    clean[retained_features].to_numpy(),
    prepared_np
)

print(
    "All-Missing Columns Removed:",
    df.isna().all().sum()
)

print("Retained Features:", len(retained_features))
print("Cleaned Shape:", clean.shape)

print(
    "Remaining Missing Values:",
    clean.isna().sum().sum()
)

print(
    "Is_Fail True Count:",
    clean["Is_Fail"].sum()
)

print("NumPy-Pandas Match:", match)

All-Missing Columns Removed: 0
Retained Features: 474
Cleaned Shape: (1567, 477)
Remaining Missing Values: 0
Is_Fail True Count: 104
NumPy-Pandas Match: True


C:\Users\maryg\AppData\Local\Temp\ipykernel_42896\342529998.py:17: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  clean["Result"] = df["Result"]
C:\Users\maryg\AppData\Local\Temp\ipykernel_42896\342529998.py:18: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  clean["Outcome"] = df["Outcome"]
C:\Users\maryg\AppData\Local\Temp\ipykernel_42896\342529998.py:20: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns a

In [ ]:
group_features = retained_features[:3]

grouped_stats = clean.groupby(
    "Outcome"
)[group_features].agg(["mean", "std"])

first_feature = retained_features[0]

numpy_sd = prepared_np[:, 0].std(ddof=1)

pandas_sd = clean[first_feature].std(ddof=1)

print(
    "Grouped Statistics:\n",
    grouped_stats.round(4)
)

print("Selected Feature:", first_feature)

print(
    "Pass Mean:",
    f"{grouped_stats.loc['Pass', (first_feature, 'mean')]:.4f}"
)

print(
    "Fail Mean:",
    f"{grouped_stats.loc['Fail', (first_feature, 'mean')]:.4f}"
)

print("NumPy SD:", f"{numpy_sd:.4f}")
print("Pandas SD:", f"{pandas_sd:.4f}")

print("SD Match:", np.isclose(numpy_sd, pandas_sd))

Grouped Statistics:
         Feature_000          Feature_001          Feature_002         
               mean      std        mean      std        mean      std
Outcome                                                               
Fail      3007.5262  88.4097   2495.1015  70.9706   2200.4419  31.3024
Pass      3014.9331  72.3133   2495.9205  80.8673   2200.5598  29.2509
Selected Feature: Feature_000
Pass Mean: 3014.9331
Fail Mean: 3007.5262
NumPy SD: 73.4808
Pandas SD: 73.4808
SD Match: True


In [ ]:
rng = np.random.default_rng(RANDOM_SEED)

feature_positions = rng.choice(
    len(retained_features),
    size=5,
    replace=False
)

assigned_features = [
    retained_features[i]
    for i in feature_positions
]

sample_size = min(
    200 + ID_SUM,
    len(clean)
)

row_positions = rng.choice(
    len(clean),
    size=sample_size,
    replace=False
)

sample = clean.iloc[row_positions][
    assigned_features +
    ["Result", "Outcome", "Is_Fail"]
].copy()

sample.insert(
    0,
    "Example_ID",
    clean.index.to_numpy()[row_positions]
)

print("Random Seed:", RANDOM_SEED)
print("Assigned Features:", assigned_features)
print("Sample Size:", len(sample))
print(sample.head())

Random Seed: 261140
Assigned Features: ['Feature_584', 'Feature_355', 'Feature_201', 'Feature_181', 'Feature_283']
Sample Size: 214
      Example_ID  Feature_584  Feature_355  Feature_201  Feature_181  \
1222        1222       0.0035       0.0356        10.35        0.540   
1542        1542       0.0037       0.0294         4.74        0.418   
190          190       0.0038       0.0239        10.02        0.511   
644          644       0.0037       0.0567         9.49        0.663   
1148        1148       0.0027       0.0262        13.42        0.514   

      Feature_283  Result Outcome  Is_Fail  
1222       2.0715      -1    Pass    False  
1542       3.1341      -1    Pass    False  
190        1.8726      -1    Pass    False  
644        2.3510      -1    Pass    False  
1148       3.3804      -1    Pass    False  


In [ ]:
feature_summary = (
    sample[assigned_features]
    .agg(["count", "mean", "std", "min", "max"])
    .T.rename_axis("Feature")
    .reset_index()
    .sort_values(by="std", ascending=False)
)

outcome_means = (
    sample.groupby("Outcome")[assigned_features]
    .mean()
    .reset_index()
)

counts = sample["Outcome"].value_counts()

pass_count = counts.get("Pass", 0)
fail_count = counts.get("Fail", 0)

failure_percentage = (
    100 * fail_count / len(sample)
)

highest_feature = feature_summary.iloc[0]["Feature"]
highest_sd = feature_summary.iloc[0]["std"]

prefix = f"{LAST_NAME}_{STUDENT_ID}"

sample_path = OUTPUT_DIR / f"{prefix}_secom_sample.csv"
summary_path = OUTPUT_DIR / f"{prefix}_feature_summary.csv"
outcome_path = OUTPUT_DIR / f"{prefix}_outcome_means.csv"
report_path = OUTPUT_DIR / f"{prefix}_analysis_report.txt"

sample.to_csv(sample_path, index=False)
feature_summary.to_csv(summary_path, index=False)
outcome_means.to_csv(outcome_path, index=False)

print("Feature Summary:\n", feature_summary.round(4))
print("Outcome Means:\n", outcome_means.round(4))
print("Highest-SD Feature:", highest_feature)
print("Highest Sample SD:", f"{highest_sd:.4f}")
print("Pass:", pass_count, "| Fail:", fail_count)

print(
    "Failure Percentage:",
    f"{failure_percentage:.2f}%"
)

print("Three CSV files exported.")

Feature Summary:
        Feature  count    mean     std     min      max
2  Feature_201  214.0  7.4940  4.8125  2.6900  58.4600
4  Feature_283  214.0  2.6396  0.7381  0.6065   4.5126
3  Feature_181  214.0  0.5473  0.2033  0.1660   1.2760
1  Feature_355  214.0  0.0353  0.0191  0.0040   0.1572
0  Feature_584  214.0  0.0041  0.0069  0.0020   0.1039
Outcome Means:
   Outcome  Feature_584  Feature_355  Feature_201  Feature_181  Feature_283
0    Fail       0.0036       0.0402       7.2667       0.5722       2.5840
1    Pass       0.0041       0.0351       7.5075       0.5459       2.6429
Highest-SD Feature: Feature_201
Highest Sample SD: 4.8125
Pass: 202 | Fail: 12
Failure Percentage: 5.61%
Three CSV files exported.


In [ ]:
loaded_sample = pd.read_csv(sample_path)
loaded_summary = pd.read_csv(summary_path)
loaded_outcomes = pd.read_csv(outcome_path)

pd.testing.assert_frame_equal(
    loaded_sample,
    sample.reset_index(drop=True),
    check_dtype=False,
    check_exact=False,
    rtol=1e-10,
    atol=1e-10
)

original_rows = clean.loc[
    loaded_sample["Example_ID"]
]

np.testing.assert_allclose(
    loaded_sample[assigned_features].to_numpy(),
    original_rows[assigned_features].to_numpy(),
    rtol=1e-10,
    atol=1e-10
)

assert np.array_equal(
    loaded_sample["Result"],
    original_rows["Result"]
)

assert np.array_equal(
    loaded_sample["Outcome"],
    original_rows["Outcome"]
)

for actual, expected in [
    (
        loaded_summary,
        feature_summary.reset_index(drop=True)
    ),
    (
        loaded_outcomes,
        outcome_means.reset_index(drop=True)
    )
]:
    pd.testing.assert_frame_equal(
        actual,
        expected,
        check_dtype=False,
        check_exact=False,
        rtol=1e-10,
        atol=1e-10
    )

report = (
    "SECOM NUMERICAL DATA ANALYSIS\n"
    f"Student: {LAST_NAME}\n"
    f"Student ID: {STUDENT_ID}\n"
    f"Random Seed: {RANDOM_SEED}\n"
    f"Verification Value: {verification_value}\n"
    f"Sample Size: {len(sample)}\n"
    f"Assigned Features: {', '.join(assigned_features)}\n"
    f"Pass Records: {pass_count}\n"
    f"Fail Records: {fail_count}\n"
    f"Sample Failure: {failure_percentage:.2f}%\n"
    f"Highest-SD Feature: {highest_feature}\n"
    f"Highest Sample SD: {highest_sd:.4f}\n"
    "Verification: PASSED\n"
)

report_path.write_text(report, encoding="utf-8")

print("Exported Records:", len(loaded_sample))
print("Original Record IDs Verified:", True)
print("Numerical Values Verified:", True)
print("Pass/Fail Outcomes Verified:", True)

for path in [
    sample_path,
    summary_path,
    outcome_path,
    report_path
]:
    print(path.name, "Created:", path.is_file())

Exported Records: 214
Original Record IDs Verified: True
Numerical Values Verified: True
Pass/Fail Outcomes Verified: True
Pena_TUPM-26-1140_secom_sample.csv Created: True
Pena_TUPM-26-1140_feature_summary.csv Created: True
Pena_TUPM-26-1140_outcome_means.csv Created: True
Pena_TUPM-26-1140_analysis_report.txt Created: True
